# 01 — Exploratory Data Analysis

Loads each language's dataset, computes class balance, text-length distributions, vocabulary stats, and writes a one-page summary into `results/tables/eda_summary.csv`.

Run after the datasets are downloaded (see `data/README.md`).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import config as C
from src.data_utils import load_dataset_for_lang

In [ ]:
summaries = []
datasets = {}
for lang in C.LANGUAGES:
    try:
        df = load_dataset_for_lang(lang)
    except FileNotFoundError as e:
        print(f'[{lang}] dataset missing: {e}'); continue
    datasets[lang] = df
    summaries.append({
        'language': lang,
        'dataset': C.DATASETS[lang].name,
        'n_rows': len(df),
        'pct_fake': float((df['label'] == 1).mean()),
        'avg_len_chars': int(df['text'].str.len().mean()),
        'median_len_chars': int(df['text'].str.len().median()),
        'p95_len_chars': int(df['text'].str.len().quantile(0.95)),
    })
summary = pd.DataFrame(summaries)
summary.to_csv(C.TABLES_DIR / 'eda_summary.csv', index=False)
summary

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 3.5), sharey=True)
if len(datasets) == 1: axes = [axes]
for ax, (lang, df) in zip(axes, datasets.items()):
    sns.histplot(df['text'].str.len().clip(upper=4000), bins=40, ax=ax)
    ax.set_title(f'{lang.upper()} — text length (chars)')
    ax.set_xlabel('chars'); ax.set_ylabel('count')
fig.tight_layout()
fig.savefig(C.TABLES_DIR / 'eda_text_lengths.png', dpi=150)
fig

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(3.5 * len(datasets), 3.5))
if len(datasets) == 1: axes = [axes]
for ax, (lang, df) in zip(axes, datasets.items()):
    counts = df['label'].value_counts().sort_index()
    ax.bar(['real', 'fake'], counts.values, color=['#4c72b0', '#c44e52'])
    ax.set_title(f'{lang.upper()} — class balance')
fig.tight_layout()
fig.savefig(C.TABLES_DIR / 'eda_class_balance.png', dpi=150)
fig

### Build the train/val/test splits and persist them

All subsequent notebooks call `get_split(lang)` which reads these parquet files. Running this cell once per language guarantees every model sees identical data.

In [ ]:
from src.data_utils import get_split
for lang in datasets:
    tr, va, te = get_split(lang, force_rebuild=True)
    print(f'{lang}: train={len(tr)} val={len(va)} test={len(te)}')